The Maxwell equations in vaccum are:
\begin{eqnarray}
  \partial_t {\bf D} &=& {\bf \nabla}\times{\bf B},\\
  \partial_t {\bf B} &=& -{\bf \nabla}\times{\bf D},\\
  {\bf \nabla}\cdot{\bf B} &=& 0,\\
  {\bf \nabla}\cdot{\bf D} &=& 0,\\
\end{eqnarray}
where ${\bf D} = {\bf E}$, $\mu = 1$, $\epsilon=1$, and $\sigma = 0$.

If a vector field ${\bf A}$ satisfies
$\partial^2_t A  - \nabla^2 {\bf A} = 0$ and ${\bf \nabla}\cdot{\bf A} = 0$,
then
${\bf E} = \partial_t {\bf A}$ and ${\bf B} = {\bf \nabla}\times {\bf A}$ satisfy the Maxwell Equations

Here we construct such solutions by taking

\begin{eqnarray}
  A_x = a_x cos(l x \pi) sin(m y \pi) sin(n z \pi) sin(\omega t)/ (l \pi),\\
A_y = a_y sin(l x \pi) cos(m y \pi) sin(n z \pi) sin(\omega t)/(m \pi),\\
A_z = -(a_x + a_y) sin(l x \pi) sin(m y \pi) cos(n z \pi) sin(\omega t)/(n \pi),
\end{eqnarray}
where $a_x$ and $a_y$ are real constants, and $l, m, n$ are positive integers, and $\omega = \pi \sqrt{l^2+m^2+n^2}$.


In [1]:
import sys
from sympy import var, Function, Derivative, Wild, Symbol, pi, symbols
from sympy import Eq, zeros, Matrix, solve, IndexedBase, Idx, ccode, cos, sin, sqrt, diff
from sympy.vector import CoordSys3D, Del, curl, divergence
import re
Cart = CoordSys3D('Cart')
delop = Del()
pi = var('pi')
t = var('t')
xx = Cart.x
yy = Cart.y
zz = Cart.z
nx = Cart.i
ny = Cart.j
nz = Cart.k


In [2]:
ax, ay, l, m, n = var(('ax', 'ay', 'l', 'm', 'n'))

Omega = sqrt(l**2 + m**2 + n**2) * pi
omega = var('omega')
Ax = ax/(l*pi)* cos(l*xx*pi)* sin(m*yy*pi)* sin(n*zz*pi) * sin(Omega*t)
Ay = ay/(m*pi)* sin(l*xx*pi)* cos(m*yy*pi)* sin(n*zz*pi) * sin(Omega*t)
Az = -(ax + ay)/ (n*pi) * sin(l*xx*pi)*sin(m*yy*pi)*cos(n*zz*pi) * sin(Omega*t)

Avec= Ax * nx + Ay * ny + Az * nz

In [4]:
Evec = - diff(Avec, t)
Bvec = curl(Avec, t)
print("Checking Maxwell's equations")
print(" div E =", divergence(Evec).simplify())
print(" div B =", divergence(Bvec).simplify())
print(" dt_E - curl B =", (diff(Evec, t) - curl(Bvec)).simplify())
print(" dt_B + curl E =", (diff(Bvec, t) + curl(Evec)).simplify())

Checking Maxwell's equations
 div E = 0
 div B = 0
 dt_E - curl B = 0
 dt_B + curl E = 0


In [13]:
x, y, z = var(('x', 'y', 'z'))

E_coor = Evec.subs({xx : x, yy : y, zz : z, sqrt(l**2 + m**2 + n**2): omega / pi})
B_coor = Bvec.subs({xx : x, yy : y, zz : z, sqrt(l**2 + m**2 + n**2): omega / pi})

Ex = E_coor.dot(nx)
Ey = E_coor.dot(ny)
Ez = E_coor.dot(nz)

Bx = B_coor.dot(nx).simplify()
By = B_coor.dot(ny).simplify()
Bz = B_coor.dot(nz).simplify()

print('void standing_wave(const int l, const int m, const int n, const double ax,\n'+
      '                   const double ay, const double x, const double y,\n'+
      '                   const double z, const double t, struct eb_st *A)')
print('{')
print('  const double omega = ', ccode(Omega), ';', sep='')

print('  A->Dx = ', ccode(Ex), ';', sep='')
print('  A->Dy = ', ccode(Ey),';', sep='')
print('  A->Dz = ', ccode(Ez), ';', sep='')

print('  A->Bx = ', ccode(Bx), ';', sep='')
print('  A->By = ', ccode(By),';', sep='')
print('  A->Bz = ', ccode(Bz), ';', sep='')
print('  A->PsiD = 0;')
print('  A->PsiB = 0;')
print('  A->rho = 0;')
print('}')



void standing_wave(const int l, const int m, const int n, const double ax,
                   const double ay, const double x, const double y,
                   const double z, const double t, struct eb_st *A)
{
  const double omega = pi*sqrt(pow(l, 2) + pow(m, 2) + pow(n, 2));
  A->Dx = -ax*omega*sin(m*pi*y)*sin(n*pi*z)*cos(omega*t)*cos(l*pi*x)/(l*pi);
  A->Dy = -ay*omega*sin(l*pi*x)*sin(n*pi*z)*cos(omega*t)*cos(m*pi*y)/(m*pi);
  A->Dz = -omega*(-ax - ay)*sin(l*pi*x)*sin(m*pi*y)*cos(omega*t)*cos(n*pi*z)/(n*pi);
  A->Bx = (-ay*pow(n, 2) - pow(m, 2)*(ax + ay))*sin(omega*t)*sin(l*pi*x)*cos(m*pi*y)*cos(n*pi*z)/(m*n);
  A->By = (ax*pow(n, 2) + pow(l, 2)*(ax + ay))*sin(omega*t)*sin(m*pi*y)*cos(l*pi*x)*cos(n*pi*z)/(l*n);
  A->Bz = (-ax*pow(m, 2) + ay*pow(l, 2))*sin(omega*t)*sin(n*pi*z)*cos(l*pi*x)*cos(m*pi*y)/(l*m);
  A->PsiD = 0;
  A->PsiB = 0;
  A->rho = 0;
}


In [7]:
Omega

pi*sqrt(l**2 + m**2 + n**2)